In [1]:
# ============================================
# MODELAGEM - PREVISÃO DE CONSUMO ENERGÉTICO
# ============================================

import pandas as pd
import numpy as np

# Carregar dataset com features
df = pd.read_csv('../data/processed/data_with_features.csv')

print("="*70)
print("📊 DATASET CARREGADO")
print("="*70)
print(f"\nLinhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")

print("\nColunas disponíveis:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

📊 DATASET CARREGADO

Linhas: 35,041
Colunas: 15

Colunas disponíveis:
   1. Date_Time
   2. Usage_kWh
   3. Lagging_Current_Reactive.Power_kVarh
   4. Lagging_Current_Power_Factor
   5. WeekStatus
   6. Day_Of_Week
   7. Load_Type
   8. Hour
   9. Month
  10. DayOfMonth
  11. is_peak_operational
  12. hour_sin
  13. hour_cos
  14. load_type_encoded
  15. is_weekend


In [2]:
df.head()

,Date_Time,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Lagging_Current_Power_Factor,WeekStatus,Day_Of_Week,Load_Type,Hour,Month,DayOfMonth,is_peak_operational,hour_sin,hour_cos,load_type_encoded,is_weekend
0,2018-01-01 00:15:00,3.17,2.95,73.21,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
1,2018-01-01 00:30:00,4.00,4.46,66.77,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
2,2018-01-01 00:45:00,3.24,3.28,70.28,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
3,2018-01-01 01:00:00,3.31,3.56,68.09,Weekday,Monday,Light_Load,1,1,1,0,0.258819,0.965926,8.625959,0
4,2018-01-01 01:15:00,3.82,4.50,64.72,Weekday,Monday,Light_Load,1,1,1,0,0.258819,0.965926,8.625959,0


In [ ]:
from sklearn.model_selection import train_test_split

print("="*70)
print("🔒 PREVENINDO DATA LEAKAGE")
print("="*70)

# ============================================
# PASSO 1: Separar TARGET e FEATURES (SEM load_type_encoded ainda)
# ============================================

TARGET = 'Usage_kWh'

# Features ORIGINAIS (sem load_type_encoded)
features_originais = [
    'Lagging_Current_Reactive.Power_kVarh',
    'Lagging_Current_Power_Factor',
    'Hour',
    'Month',
    'DayOfMonth',
    'is_peak_operational',
    'hour_sin',
    'hour_cos',
    'is_weekend',
    'Load_Type'  # ← Vamos manter texto por enquanto
]

X_original = df[features_originais]
y = df[TARGET]

# ============================================
# PASSO 2: DIVIDIR primeiro (ANTES de fazer encoding)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_original, y,
    test_size=0.2,
    random_state=42
)

print(f"\n✅ Dados divididos:")
print(f"   Treino: {len(X_train):,} linhas")
print(f"   Teste:  {len(X_test):,} linhas")

# ============================================
# PASSO 3: Calcular encoding APENAS com TREINO
# ============================================

# Calcular médias usando APENAS dados de treino
train_data = pd.concat([X_train, y_train], axis=1)
medias_treino = train_data.groupby('Load_Type')['Usage_kWh'].mean()

print(f"\n📊 Médias calculadas (APENAS treino):")
for load_type, media in medias_treino.items():
    print(f"   {load_type:15s}: {media:.2f} kWh")

# ============================================
# PASSO 4: Aplicar encoding em TREINO e TESTE
# ============================================

# Treino
X_train['load_type_encoded'] = X_train['Load_Type'].map(medias_treino)

# Teste (usa encoding calculado no treino!)
X_test['load_type_encoded'] = X_test['Load_Type'].map(medias_treino)

# Remover coluna Load_Type (não precisamos mais)
X_train = X_train.drop('Load_Type', axis=1)
X_test = X_test.drop('Load_Type', axis=1)

print(f"\n✅ Encoding aplicado SEM data leakage!")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape:  {X_test.shape}")


🔒 PREVENINDO DATA LEAKAGE

✅ Dados divididos:
   Treino: 28,032 linhas
   Teste:  7,009 linhas

📊 Médias calculadas (APENAS treino):
   Light_Load     : 8.61 kWh
   Maximum_Load   : 59.32 kWh
   Medium_Load    : 38.32 kWh

✅ Encoding aplicado SEM data leakage!
   X_train shape: (28032, 10)
   X_test shape:  (7009, 10)


#Primeiro Modelo de Regressão Linear simples (Colocando uma Linha de base)

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("="*70)
print("🤖 MODELO 1: LINEAR REGRESSION (Baseline)")
print("="*70)

#treino
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
#previsões
y_pred = model_lr.predict(X_test)

#métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_absolute_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("\n📊 RESULTADOS:")
print(f"   MAE:  {mae:.2f} kWh")
print(f"   RMSE: {rmse:.2f} kWh")
print(f"   R²:   {r2:.4f}")
print(f"   MAPE: {mape:.2f}%")

print("\n✅ Baseline estabelecido!")

🤖 MODELO 1: LINEAR REGRESSION (Baseline)

📊 RESULTADOS:
   MAE:  7.82 kWh
   RMSE: 2.80 kWh
   R²:   0.8924
   MAPE: 84.72%

✅ Baseline estabelecido!


#Modelo 2 (Random Forest)

In [9]:
from sklearn.ensemble import RandomForestRegressor

print("="*70)
print("🌳 MODELO 2: RANDOM FOREST")
print("="*70)

# Treinar
model_rf = RandomForestRegressor(
    n_estimators=100,    # 100 árvores
    random_state=42,
    n_jobs=-1            # Usar todos os cores
)

print("Treinando Random Forest...")
model_rf.fit(X_train, y_train)

# Prever
y_pred_rf = model_rf.predict(X_test)

# Métricas
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

print("\n📊 RESULTADOS:")
print(f"   MAE:  {mae_rf:.2f} kWh")
print(f"   RMSE: {rmse_rf:.2f} kWh")
print(f"   R²:   {r2_rf:.4f}")
print(f"   MAPE: {mape_rf:.2f}%")

# Comparar com baseline
print(f"\n📈 MELHORIA vs Baseline:")
print(f"   MAE:  {((mae - mae_rf)/mae)*100:+.1f}%")
print(f"   R²:   {((r2_rf - r2)/r2)*100:+.1f}%")

🌳 MODELO 2: RANDOM FOREST
Treinando Random Forest...

📊 RESULTADOS:
   MAE:  0.44 kWh
   RMSE: 1.34 kWh
   R²:   0.9984
   MAPE: 3.64%

📈 MELHORIA vs Baseline:
   MAE:  +94.4%
   R²:   +11.9%
